# ET1 Pilot: SmolLM2-360M-Instruct

**Experiment:** Does Semantic Annotation Structure in Training Data Affect Model Calibration and Uncertainty Awareness?

**This notebook** runs all 7 conditions × 3 seeds = 21 training runs for SmolLM2-360M-Instruct.

**Runtime:** Set to **GPU** (T4 minimum, A100 preferred). Estimated time: ~4-6 hours.

**Checkpointing:** Built in. If Colab disconnects, re-run from Cell 3 onward — completed runs are skipped automatically.

## 1. Environment Setup

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available — set Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
import os

REPO_URL = "https://github.com/jemsbhai/jsonld-ex-experiments.git"
REPO_DIR = "/content/jsonld-ex-experiments"
ET1_DIR = os.path.join(REPO_DIR, "ET1")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull

os.chdir(ET1_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q -r requirements.txt

import transformers, peft, datasets
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"datasets: {datasets.__version__}")

## 2. Mount Google Drive (for persistent results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RESULTS = "/content/drive/MyDrive/ET1_results/smollm2-360m"
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f"Results will be saved to: {DRIVE_RESULTS}")

## 3. Log Environment (Reproducibility)

In [ ]:
import json
import platform
from datetime import datetime, timezone

env_info = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "os": platform.platform(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "peft_version": peft.__version__,
    "cuda_version": torch.version.cuda or "none",
}

env_path = os.path.join(DRIVE_RESULTS, "environment.json")
with open(env_path, "w") as f:
    json.dump(env_info, f, indent=2)

print(json.dumps(env_info, indent=2))

## 4. Generate Knowledge Base

In [ ]:
import yaml
from src.run_pilot import load_config, generate_and_split_kb

config = load_config("configs/training_config.yaml")

config["paths"]["data_dir"] = "data"
config["paths"]["results_dir"] = DRIVE_RESULTS
config["paths"]["checkpoints_dir"] = "checkpoints"

kb_path = os.path.join(config["paths"]["data_dir"], "meridian_kb.json")
if os.path.exists(kb_path):
    print(f"KB already exists at {kb_path}, loading...")
    from src.knowledge_base import load_knowledge_base, split_knowledge_base
    from src.fact import Fact
    kb = load_knowledge_base(kb_path)
    raw_splits = split_knowledge_base(kb, config["knowledge_base"])
    splits = {name: [Fact.from_meridian(f) for f in facts] for name, facts in raw_splits.items()}
else:
    splits = generate_and_split_kb(config["knowledge_base"], config["paths"]["data_dir"])

for name, facts in splits.items():
    print(f"  {name}: {len(facts)} facts")

## 5. Run Pilot — SmolLM2-360M-Instruct

All 7 conditions × 3 seeds = 21 runs. Checkpointing is automatic.

In [ ]:
import logging, sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)

config["pilot_models"] = [
    m for m in config["pilot_models"]
    if m["short_name"] == "smollm2-360m"
]

assert len(config["pilot_models"]) == 1
print(f"Model: {config['pilot_models'][0]['name']}")
print(f"Conditions: {[c['id'] for c in config['conditions']]}")
print(f"Seeds: {config['training']['seeds']}")
print(f"Total runs: {len(config['conditions']) * len(config['training']['seeds'])}")

In [ ]:
from src.run_pilot import (
    build_run_plan, is_run_complete, run_single_condition,
    print_summary, compare_results
)

plan = build_run_plan(config)
results_dir = config["paths"]["results_dir"]
all_results = {}

for entry in plan:
    if is_run_complete(entry["run_key"], results_dir):
        print(f"SKIP (already complete): {entry['run_key']}")
        result_path = os.path.join(results_dir, f"{entry['run_key']}.json")
        with open(result_path) as f:
            all_results[entry["run_key"]] = json.load(f)

for entry in plan:
    if entry["run_key"] in all_results:
        continue

    print(f"\n{'='*60}")
    print(f"Starting: {entry['run_key']}")
    print(f"{'='*60}")

    result = run_single_condition(
        model_name=entry["model_name"],
        model_short_name=entry["model_short_name"],
        condition=entry["condition"],
        seed=entry["seed"],
        train_facts=splits["train"],
        val_facts=splits["val"],
        test_facts={"test_id": splits["test_id"], "test_ood": splits["test_ood"]},
        config=config,
    )
    all_results[entry["run_key"]] = result
    torch.cuda.empty_cache()

print(f"\nAll {len(plan)} runs complete.")

## 6. Results Summary

In [ ]:
print_summary(all_results)

conditions_present = {r.get("condition") for r in all_results.values()}
if "C1" in conditions_present and "C4" in conditions_present:
    print("\n--- C1 (plain text) vs C4 (jsonld-ex full) ---")
    for split in ["test_id", "test_ood"]:
        print(f"\n  Split: {split}")
        comp = compare_results(all_results, baseline="C1", treatment="C4", split=split)
        for metric, vals in comp.items():
            delta_str = f"{vals['delta']:+.4f}" if vals["delta"] is not None else "N/A"
            print(f"    {metric:<30} Δ = {delta_str}")

In [ ]:
import glob

result_files = sorted(glob.glob(os.path.join(DRIVE_RESULTS, "*.json")))
print(f"Result files on Google Drive ({len(result_files)}):")
for f in result_files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f)} ({size_kb:.1f} KB)")